In [1]:
from ultralytics import YOLO
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import cv2
import random
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_PATH = "../BoneFractureYolo8"

train_images = os.path.join(BASE_PATH, "train", "images")
train_labels = os.path.join(BASE_PATH, "train", "labels")

val_images = os.path.join(BASE_PATH, "valid", "images")
val_labels = os.path.join(BASE_PATH, "valid", "labels")

test_images = os.path.join(BASE_PATH, "test", "images")
test_labels = os.path.join(BASE_PATH, "test", "labels")

In [3]:
import yaml

with open(os.path.join(BASE_PATH, "data.yaml")) as f:
    data = yaml.safe_load(f)

print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 7, 'names': ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive'], 'roboflow': {'workspace': 'veda', 'project': 'bone-fracture-detection-daoon', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/veda/bone-fracture-detection-daoon/dataset/4'}}


In [4]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=os.path.join(BASE_PATH, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project="D:/yolo_results",
    name="train"
)

Ultralytics 8.4.21  Python-3.12.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../BoneFractureYolo8\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [5]:
run_dir = model.trainer.save_dir
print("Training results saved in:", run_dir)

Training results saved in: D:\yolo_results\train


In [6]:
metrics = model.val(
    data=os.path.join(BASE_PATH, "data.yaml"),
    split="test"
)

print(metrics)

Ultralytics 8.4.21  Python-3.12.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 2.00.7 MB/s, size: 11.3 KB)
val: Scanning C:\Users\Admin\Desktop\graduate study\BoneFractureYolo8\test\labels... 169 images, 86 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 169/169 791.1it/s 0.2s0.0s
val: New cache created: C:\Users\Admin\Desktop\graduate study\BoneFractureYolo8\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 6.3it/s 1.8s.2s
                   all        169         96      0.209      0.303      0.192      0.066
        elbow positive         13         17      0.101      0.176     0.0542      0.017
      fingers positive         22         27      0.286       0.37      0.236     0.0644
      forearm fracture         13         14      0.265      0.214      0.219     

In [7]:
results = model.predict(
    source=test_images,
    conf=0.25,
    save=True
)


image 1/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\distal-humerus-fracture-1_jpg.rf.831cb137cfcbde1079f86abd5f5f2867.jpg: 640x256 1 elbow positive, 35.3ms
image 2/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_0_png.rf.99862308d714bff3f9c410adf5ca93ac.jpg: 480x640 1 fingers positive, 92.8ms
image 3/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1000_png.rf.a53c5e186c03961bf88075c6e3e94cf6.jpg: 544x640 (no detections), 40.6ms
image 4/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1015_png.rf.3b7320c3c40771fa5532bf713a728b83.jpg: 544x640 (no detections), 39.1ms
image 5/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1015_png.rf.9181f8eb07451331e22381bacb3a5bd2.jpg: 640x640 (no detections), 32.5ms
image 6/169 C:\Users\Admin\Desktop\graduate study\Deep

In [8]:
conf_matrix_path = os.path.join(run_dir, "confusion_matrix.png")

if os.path.exists(conf_matrix_path):
    img = mpimg.imread(conf_matrix_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()
else:
    print("confusion_matrix.png not found.")

<Figure size 800x800 with 1 Axes>

In [9]:
conf_matrix_norm_path = os.path.join(run_dir, "confusion_matrix_normalized.png")

if os.path.exists(conf_matrix_norm_path):
    img = mpimg.imread(conf_matrix_norm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Normalized Confusion Matrix")
    plt.show()
else:
    print("confusion_matrix_normalized.png not found.")

<Figure size 800x800 with 1 Axes>

In [10]:
import pandas as pd

results_csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    print(df_results.columns.tolist())
    display(df_results.head())
else:
    print("results.csv not found.")

['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,34.2887,2.78482,7.50826,2.21996,0.00102,0.46637,0.00360,0.00071,2.83110,3.77996,2.15928,0.000302,0.000302,0.000302
1,2,62.1579,2.53384,5.68114,2.00331,0.35510,0.06874,0.01750,0.00454,2.53302,3.73389,2.07797,0.000593,0.000593,0.000593
2,3,89.3931,2.48906,4.77479,2.00389,0.24267,0.03771,0.02981,0.00814,2.47668,3.73775,2.02640,0.000872,0.000872,0.000872
3,4,117.6320,2.46554,4.14707,2.01096,0.27509,0.08015,0.05645,0.01958,2.39820,3.47201,2.16896,0.000855,0.000855,0.000855
4,5,145.2000,2.39745,3.74997,1.96612,0.48804,0.12041,0.10068,0.03266,2.44303,3.35711,2.04500,0.000837,0.000837,0.000837


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: losses
if "train/box_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/box_loss"], label="train box")
if "train/cls_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/cls_loss"], label="train cls")
if "train/dfl_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/dfl_loss"], label="train dfl")
if "val/box_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/box_loss"], label="val box", linestyle="--")
if "val/cls_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/cls_loss"], label="val cls", linestyle="--")
if "val/dfl_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/dfl_loss"], label="val dfl", linestyle="--")

axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Right: metrics
if "metrics/precision(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/precision(B)"], label="precision")
if "metrics/recall(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/recall(B)"], label="recall")
if "metrics/mAP50(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/mAP50(B)"], label="mAP50")
if "metrics/mAP50-95(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/mAP50-95(B)"], label="mAP50-95")

axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Value")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<Figure size 1400x500 with 2 Axes>